# `DDict` Tutorial: Sharing Data Across Languages and Processes

**Estimated time:** ~25 minutes  
**Format:** 6 short exercises with hidden solutions.

This notebook introduces several patterns that can be used when working with diverse types of processes that need to share data. The notebook demonstrates how to start Python processes, share data by serializing and attaching to a Dragon Distributed Dictionary, including how to share data between Python and C++. These same ideas can also be supplied to MPI programs. 

Several concepts from the Distributed Dictionary are covered including the wait_for_keys option, checkpointing, and batch put. These concepts can help with synchronization in your program based on data availability and with placing data close to compute. All the patterns presented here apply equally to Dragon Process Groups and to MPI programs in general.

All of the Dragon interaction in this tutorial is incorporated into a stand-alone program (not a notebook) in this same directory called [parallel_pi.py](parallel_pi.py) and some associated C++ located in [worker.cpp](worker.cpp). This tutorial develops the background to understand the [parallel_pi.py](parallel_pi.py) program and to help the reader build on these concepts. 

## Setup

To begin the tutorial some set up is in order. Execute the following cell to do some imports and get Dragon ready for the rest of this tutorial.


In [1]:
import os
import sys
from pathlib import Path
import multiprocessing as mp
import random

import dragon

try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

import dragon.native as dn
from dragon.data import DDict
from dragon.native.process import ProcessTemplate
from dragon.native.process_group import ProcessGroup
from dragon.utils import XPickler

TUTORIAL_DIR = Path.cwd()
print(f"Dragon ready in {TUTORIAL_DIR}")

Dragon ready in /workspaces/pearc_2026_tutorial/course2/sharing_data_mpi_and_others


---

## Exercise 1 - DDict Serialize and Attach

### Background:  

Dragon objects can be serialized. These serialized representations can be passed between processes and used to re-attached in the receiving process. The following three examples serialize a `DDict` in one process and attaches to it in another using three modes of object passing. Multiprocessing provides the highest level approach where serialization and attach happen automatically as presented in mode 1. Using the other two modes, the serialization and the attach are done explicitly.  

#### Mode 1: Multiprocessing Process Creation and Argument Passing Style
[The introductory DDict notebook](../../course1/managing_data_with_ddict/ddict_tutorial.ipynb) demonstrated that you can pass Dragon objects between processes as *arguments*. Multiprocessing makes it possible to pass arguments and call functions in new processes. The abstraction works well for when writing multiprocessing Python code.

- `ddict` is passed as an argument to `worker`
- `dd` is provided in `args` in the `ProcessTemplate`

```python
def worker(dd):
    dd["hello"] = "world"
    # carry on from here
    
dd = DDict(1, 1, 64 * 1024 * 1024)

# ProcessGroup is a Dragon only construct. Pool would be the multiprocessing equivalent.
pg = ProcessGroup() 

num_procs = 10
proc_template = ProcessTemplate(target=worker, args=(dd,))
pg.add_process(nproc=num_procs, template=proc_template)
pg.init()
pg.start()

# Rest of orchestrator logic

pg.join()
pg.close()
dd.destroy()
```

#### Mode 2: Serialize/Attach Argument Passing Style

While the previous abstraction is very convenient, there are situations where there may be less control 
over how the processes get started. For such situations it may be necessary to serialize the Dragon
object and then attach to it in the child. The code below shows this pattern. The process template 
provides a serialized repesentation of the object and the child attaches using the serialized descriptor.

- `ProcessTemplate` contains `args` with the serialized DDict `dd.serialize()`
- A serialized DDict is a string representation that contains all the information needed to attach to the DDict.
- `worker` receives the `dd_ser` argument as a string and uses it to attach to the DDict using `DDict.attach`

```python
def worker(dd_ser):
    dd = DDict.attach(dd_ser)
    dd["hello"] = "world"
    # carry on from here
    
dd = DDict(1, 1, 64 * 1024 * 1024)
serialized_ddict = dd.serialize()

pg = ProcessGroup()

num_procs = 10
proc_template = ProcessTemplate(target=worker, args=(serialized_ddict,))
pg.add_process(nproc=num_procs, template=proc_template)
pg.init()
pg.start()

# Rest of orchestrator logic

pg.join()
pg.close()
dd.destroy()
```

#### Mode 3: Serialize/Attach with Command-line Arguments

Moving further away from multiprocessing's ability to pass arguments to functions, It might be the case
that you need to pass arguments on the command-line to some other process. In that case you start the 
executable and provide the command-line arguments you want to pass to the code. A serialized 
DDict can be passed on the command-line to a child process. The child accesses it 
through the command-line arguments like any other program.

- In this case a program module (not a notebook) would contain the *worker* code. The worker is a stand-alone program that gets its serialized DDict as a command-line argument. 
- The `ProcessTemplate` starts the program's executable.
- The `args` are command-line arguments in this case.
- Once the worker program is started it can examine the command-line arguments to get the serialized DDict and atttach to it.

##### Worker Program

Here is the *worker* program for mode 3 where the serialized DDict is passed as a command-line argument. Assume this file is saved as worker.py.

```python
import dragon
from dragon.data import DDict

def main():
    dd_ser = sys.argv[1]
    dd = DDict.attach(dd_ser)
    dd["hello"] = "world"
    # carry on from here

if __name__ == "__main__":
    main()
```

##### Orchestration Program

Here is a snippet of parent process code that starts the *worker* program.

```python

dd = DDict(1, 1, 64 * 1024 * 1024)
serialized_ddict = dd.serialize()

pg = ProcessGroup()

# In this example there would have to be logic that looks for "worker" on the command-line in 
# the child process and then calls the worker function. That code is not shown here.
num_procs = 10
script_path = TUTORIAL_DIR / "worker.py"
proc_template = ProcessTemplate(target=sys.executable, # sys.executable is python in this case
            args=(str(script_path), serialized_ddict),
            cwd=os.getcwd())

pg.add_process(nproc=num_procs, template=proc_template)
pg.init()
pg.start()

# Rest of logic
pg.join()
pg.close()
d.destroy()
```

### Your Task:

Use the Serialize/Attach Argument Passing Style (second of the patterns) to write this code.

1. Create a `DDict` with 1 manager, 1 node, and 64 MB.
2. Write `worker(serialized_ddict)`.
3. In the worker, attach to the `DDict`, use `fetch_add("worker_id")`, and store `f"hello from worker {worker_id}"` under `f"worker_{worker_id}"`.
4. Launch 3 Dragon native Python processes, print the results in the main program, and destroy the dictionary.

In [ ]:
# -- Exercise 1 --------------------------------------------------------------

def worker(serialized_ddict):
    # Write the child code here
    pass


# Write the orchestrator code here

<details>
<summary><b>▶ Show Answer</b></summary>

```python
def worker(serialized_ddict):
    d = DDict.attach(serialized_ddict)
    id = d.fetch_add("worker_id")
    d[f"worker_{id}"] = f"hello from worker {id}"

d = DDict(1, 1, 64 * 1024 * 1024)
serialized_ddict = d.serialize()
pg = ProcessGroup()

num_procs = 3
proc_template = ProcessTemplate(target=worker, args=(d.serialize(),))
pg.add_process(nproc=num_procs, template=proc_template)
pg.init()
pg.start()
pg.join()
pg.close()

for i in range(num_procs):
    val = d[f"worker_{i}"] # The data will be there because it is after the join.
    print(f"From worker {i}:", val, flush=True)
    
d.destroy()
```

</details>

---

## Exercise 2 - Efficient Data Distribution

### Background:

When every rank or process needs the same data to do its work, and when these processes/ranks are scaled to large scales across nodes, it may be desirable to efficiently distribute that data to all the 
processes/ranks in the program. In that case, the `bput` DDict operation can be used to distribute data to all managers in the DDict. If these managers are distributed across all nodes, then every node will have an in-memory copy
of the `bput` data. An individual rank can then get that data node-locally by using `bget` to retrieve it. `bput` stands for *broadcast put* and it is used for the efficient broadcasting of data to all managers of the 
DDict. 

By using `wait_for_keys=True` the ranks can do a `bget` operation as soon as they are started without having to synchronize with the parent/orchestrating process around data availability. Specifying `wait_for_keys`
means that the DDict will not raise an error if the key does not exist. Instead it will efficiently wait for the key and then respond once the key is available. 

#### Orchestrator Code:

Consider this example of using `bput`. Imagine for a moment that all your ranks needed a copy of a model that they could then use as part of a simulation/inference loop. 
```python

import dragon
from dragon.data.ddict import DDict
import torch
import torch.nn as nn
from modeldef import YourModelClass

num_nodes = # the number of nodes in your allocation

dd = DDict(num_nodes, 1, num_nodes * 64 * 1024 * 1024, wait_for_keys=True, working_set_size=2) 

# Recreate the exact same model architecture class
model = YourModelClass(*args, **kwargs)

# Load the state dictionary (include weights_only=True for security)
state_dict = torch.load('model.pth', weights_only=True)

dd.bput("model", state_dict) # copied to every node using a tree dispersal

# When dd is no longer needed
dd.destroy()
```

#### Rank/Process Code:

Then, in every rank, you can load the `state_dict` from local memory. 
```python
# Executed within each rank/process.

# DDict is attached
dd = DDict.attach(dd_ser)

# Retrieve the state_dict locally
state_dict = dd.bget("model")

# Load the parameters into your model
model.load_state_dict(state_dict)

# Use the model for inference
model.eval()

# When rank done with dd 
dd.detach()
```

### Your Task:

1. Create a `DDict` with `wait_for_keys=True` and `working_set_size=4`.
2. Write `worker(serialized_ddict)` that attaches to a dictionary, gets a process id with `fetch_add("worker_id")`, reads an integer value using `bget`, then writes worker{process_id} as a key and the value x process_id as its value.
3. Launch 4 Dragon native Python workers with a `ProcessGroup`.
4. In the parent, publish the broadcast value using `bput`.
5. Have the parent access all the values written by the workers and add them all together and report the sum. 
6. Make sure all processes that `attach` also call `detach` when done.

In [ ]:
# -- Exercise 2 --------------------------------------------------------------

def worker(serialized_ddict):
    # Write the worker code here
    pass


# Write the orchestrator code here


<details>
<summary><b>▶ Show Answer</b></summary>

```python
def worker(serialized_ddict):
    d = DDict.attach(serialized_ddict)
    proc_id = d.fetch_add("worker_id")
    shared_int = d.bget("shared")
    d[f"worker{proc_id}"] = shared_int * proc_id
    d.detach()

num_workers = 4
d = DDict(1, 1, 128 * 1024 * 1024, wait_for_keys=True, working_set_size=4)
pg = ProcessGroup()
template = ProcessTemplate(target=worker, args=(d.serialize(),))
pg.add_process(nproc=num_workers, template=template)

pg.init()
pg.start()
d.bput("shared", 42) # Workers will wait until this appears
total = 0
for worker_id in range(num_workers):
    total += d[f"worker{worker_id}"] # this will wait until value is available

pg.join()
pg.close()
print(f"final total = {total}")
d.destroy()
```

</details>

---

## Exercise 3 - Launch a Python program with `sys.executable`

### Background:

The next step is to launch a separate Python program using `ProcessTemplate(target=sys.executable, ...)` and pass the serialized DDict argument on the command line.

### Your Task:

1. Write a worker program to file `py_worker_ex3.py` in the next cell.  DO NOT delete the writefile first line. 
2. Read the serialized `DDict` from `sys.argv[1]`.
3. Attach, fetch a unique id, and store `f"Hello from proc {proc_id}"` under key `proc_id`.
4. Once you have written the program, execute the cell to write the contents of it to the given file name.
5. In the Orchestrator code cell, launch 4 workers with a `ProcessGroup`, print the values, and destroy the dictionary. Be sure to use the example code from the third means of starting processes where arguments are passed on the command-line to a new executable. 

#### Worker Code Here - After writing your code, execute this cell to write the file.

In [ ]:
%%writefile py_worker_ex3.py

#Write the main program of the worker here.

#### Orchestrator Code Here

In [15]:
# -- Exercise 3 --------------------------------------------------------------
from dragon.data import DDict

script_path = TUTORIAL_DIR / "py_worker_ex3.py"

# Rest of Orchestrator code here

<details>
<summary><b>▶ Show Answer</b></summary>

#### py_worker_ex3.py contents

```python
import sys
from dragon.data import DDict
def main():
    d_ser = sys.argv[1]
    dd = DDict.attach(d_ser)
    proc_id = dd.fetch_add("worker")
    dd[proc_id] = f"Hello from proc {proc_id}"
    print(f"Worker {proc_id} exiting", flush=True)

if __name__ == '__main__':
    main()
```

#### Main Program of Orchestrator

```python
from dragon.data import DDict

script_path = TUTORIAL_DIR / "py_worker_ex3.py"
d = DDict(1, 1, 64 * 1024 * 1024, working_set_size=2, wait_for_keys=True)
pg = ProcessGroup()
template = ProcessTemplate(
    target=sys.executable,
    args=(str(script_path), d.serialize()),
    cwd=os.getcwd(),
)
pg.add_process(nproc=4, template=template)
pg.init()
pg.start()

for worker_id in range(4):
    print(d[worker_id], flush=True) # Will wait here until data is available
    
pg.join()
pg.close()
d.destroy()
print("All Done")
```

</details>

---

## Exercise 4 - Launch a compiled C++ program and share data between C++ workers

### Background:
The C++ path in [parallel_pi.py](parallel_pi.py) launches a compiled worker binary with the same `ProcessTemplate` interface. The [worker.cpp](worker.cpp) example code shows how `DDict<Serializable, Serializable>` and `fetch_add` are used from C++. In this exercise we'll duplicate some of that behavior and allow you to customize it for your own needs/investigation. You will learn how to attach to a DDict in C++ and share data, via the DDict, amongst C++ processes.

**CAVEAT**: Given that this exercise incorporates C++, if you are NOT running in a container you will need to supply your own C++ compiler and may need to slightly modify the compile command below. If you are running in a container, then this notebook introduction should work for you as the container includes the GNU C++ compiler.

In C++, Dragon defines a class called *Serializable* that allows all types of serializable data to be stored in and retrieved from a Distributed Dictionary. When a C++ DDict is attached, it can be instantiated using this *Serializable* class to serialize and deseriailize keys and values automatically. For instance, consider this code.

```c++
// This attaches ddict using the Serializable class for keys and values.
// nullptr is a timeout which in this case means *don't timeout ever*.

DDict<Serializable, Serializable> ddict(ser_ddict, nullptr); 
std::string key_str = "my_key";
std::string value_str = "my_value";
// std::string can be automatically converted to a SerializableString
SerializableString key = key_str; 
SerializableString value = value_str;
ddict[key] = value;

// likewise you can retrieve a value for a key
SerializableString retrieved_value = ddict[key];
std::cout << "Here is the retrieved value: " << retrieved_value.getVal() << std::endl; 
```

The code above demonstrates how a serialized DDict can be attached and C++ and a key/value pair can be stored in the DDict. There are other common Serializable types already defined in Dragon including:

- SerializableInt
- SerializableDouble
- SerializableString
- SerializableIntVector
- SerializableDoubleVector
- Serializable2DIntMatrix
- Serializable2DDoubleMatrix
- SerializableByteBuffer

Other types may be defined by Dragon in the future as well. It is also possible to define your own Serializable types by following the patterns seen in the [serializable.hpp include file](https://github.com/DragonHPC/dragon/blob/main/src/include/dragon/serializable.hpp). 

### Your Task:

1. Fill in the missing C++ lines below. 
2. Retrieve the neighbor's value given the neighbor's key in your worker C++ program. **HINT:** To get a neighbor's value you can compute the neighbor id using neighbor_id = (worker_id + 1) % 3 which will give you the next neighbor's id and then you can build the neighbor's key using the example code above as a pattern.
3. Once you have the neighbor's value in the C++ worker code, print the neighbor's value.
4. Once your C++ code is written, run the cell it is in to write the file.
5. Then compile the file by running the g++ command in the following cell.
6. Finally run the Python cell to launch 3 workers from Python, wait for their exit, and destroy the ddict. *You do not need to modify the Python code in this exercise. Just run it*

In [ ]:
%%writefile worker_ex4.cpp
#include <iostream>
#include <string>
#include <dragon/dictionary.hpp>
#include <dragon/serializable.hpp>

int main(int argc, char* argv[]) {
    try {
        const char* ser_ddict = argv[1];
        DDict<Serializable, Serializable> ddict(ser_ddict, nullptr);
        // TODO: initialize the worker_id using fetch_add.
        int worker_id = 0;
        std::string key_str = "worker" + std::to_string(worker_id);
        std::string value_str = "cpp-" + std::to_string(worker_id);
        // TODO: store value_str in the DDict at the key_str and then get the neighbor value and print it.
        std::cout << "Worker " << worker_id << " finished." << std::endl;
    } catch (DragonError ex) {
        std::cout << "Got error " << ex << std::endl;
        return 1;
    }
    return 0;
}

In [6]:
!g++ $(dragon-config -o) -I ${VIRTUAL_ENV:-$CONDA_PREFIX}/include -std=c++14 -o worker_ex4 worker_ex4.cpp $(dragon-config -l) -ldl

In [ ]:
binary_path = TUTORIAL_DIR / "worker_ex4"
d = DDict(1, 1, 64 * 1024 * 1024, working_set_size=2, wait_for_keys=True)
pg = ProcessGroup()
template = ProcessTemplate(target=str(binary_path), args=(d.serialize(),), cwd=os.getcwd())
pg.add_process(nproc=3, template=template)
try:
    pg.init()
    pg.start()
    pg.join()
finally:
    try:
        pg.close()
    except:
        pass 
    d.destroy()

<details>
<summary><b>▶ Show Answer</b></summary>

```cpp
#include <iostream>
#include <string>
#include <dragon/dictionary.hpp>
#include <dragon/serializable.hpp>

int main(int argc, char* argv[]) {
    try {
        const char* ser_ddict = argv[1];
        DDict<Serializable, Serializable> ddict(ser_ddict, nullptr);
        // TODO: initialize the worker_id using fetch_add.
        int worker_id = ddict.fetch_add("worker_id");
        std::string key_str = "worker" + std::to_string(worker_id);
        std::string value_str = "cpp-" + std::to_string(worker_id);
        // TODO: store value_str in the DDict.
        SerializableString key = key_str;
        SerializableString value = value_str;
        ddict[key] = value;
        int neighbor_id = (worker_id + 1) % 3;
        std::string neighbor_key_str = "worker" + std::to_string(neighbor_id);
        SerializableString neighbor_key = neighbor_key_str;
        SerializableString neighbor_val = ddict[neighbor_key];
        std::cout << "Got " << neighbor_val.getVal() << " from neighbor" << std::endl;
        std::cout << "Worker " << worker_id << " finished." << std::endl;
    } catch (DragonError ex) {
        std::cout << "Got error " << ex << std::endl;
        return 1;
    }
    return 0;
}
```
</details>

---

## Exercise 5 - Checkpointing and Parallel Execution

### Background: 

Because of its usefulness, this exercise covers checkpointing in the DDict. The DDict can be constructed with a working set of key/value pairs. When a DDict is constructed, the allowable maximum size of the working set can be specified using the `working_set_size` keyword argument. This specifies the maximum number of active checkpoints that are available in the DDict at any moment during its lifetime. 

A checkpoint is a copy of the non-persistent keys that are stored in the DDict. Think of the working set as a sliding window of allowable checkpoints. Each DDict manager maintains its own sliding window of checkpoints, but by their nature most managers will be in sync with each other. Each checkpoint is assigned an id. For instance, the first checkpoint is checkpoint 0, second is 1, and so on.

Clients maintain a checkpoint id. Each client starts with a checkpoint id of 0. This means that all puts/writes of non-persistent keys are written into checkpoint 0 by all clients unless a client calls `checkpoint` which then increments their checkpoint id. Puts/writes done after a call to `checkpoint` are written into the next checkpoint copy. In this way old values for key/value pairs can be left in the DDict while new values are also being written. This allows values that are being used for computation to safely change over time without changing in the middle of some computation. Newly computed values for key/value pairs can be written to a separate, new checkpoint.

Consider this worker function that takes care of smoothing segments of a function. This is from the DDict [function smoothing example](https://github.com/DragonHPC/dragon/blob/main/examples/dragon_data/ddict/ddict_smoothing.py).

```python
def worker_smooth(ddict: DDict, chunk_id: int, iterations: int, checkpoint_id: int, trace: bool) -> None:
    ddict.checkpoint_id = checkpoint_id
    num_chunks = ddict["num_chunks"]
    # Load the initial chunk data for this worker
    current = np.array(ddict[f"chunk_{chunk_id}"])

    for step in range(iterations):
        left_val = None
        right_val = None

        # [ start-data-sharing-ref ]
        if chunk_id > 0:
            left_segment = np.array(ddict[f"chunk_{chunk_id - 1}"])
            left_val = left_segment[-1]
        if chunk_id < num_chunks - 1:
            right_segment = np.array(ddict[f"chunk_{chunk_id + 1}"])
            right_val = right_segment[0]
        # [ end-data-sharing-ref ]

        # [start-compute-avg]
        current = smooth_segment(current, left_val, right_val)

        ddict.checkpoint()

        ddict[f"chunk_{chunk_id}"] = current
        # [end-compute-avg]

    ddict.detach()
```

The function smooths a segment of a function. It does this by considering each point within a segment and averaging it with its neighbors computing a new segment. This works fine except at boundaries where a neighbor is in a neighboring segment. But because workers are averaging segments in parallel, for the algorithm to work correctly, all workers need to work with the same version of the segments. 

This is managed with checkpoints in the code. All workers read the same time step of the algorithm in the same checkpoint. When a new segment has been computed, it is stored in a new checkpoint by first calling checkpoint, and then storing the newly computed segment.

There are two important things to point out. First, this abstraction of checkpoints works well when *versions* of data are necessary in a program. Secondly, this idea of a working set means that each worker/rank can work independently and perhaps a little ahead or behind its neighbors. In the code above the two lines

```python
left_segment = np.array(ddict[f"chunk_{chunk_id - 1}"])
# and
right_segment = np.array(ddict[f"chunk_{chunk_id + 1}"])
```

very naturally guarantee the parallel workers will not get to far out of sync because those two lines will block, waiting for the 
respective values to become available in the given checkpoint. Constructing a DDict with `wait_for_keys=True` is important when using checkpointing because it means the DDict will efficiently provide synchronization of your workers at just the right time in your code.

### Your Task:

Call checkpoint in the *pi* calculation program in both the worker function and in the main program. 

1. The worker repeatedly computes versions of *pi* as for a given number of iterations. On each iteration it computes a new version of *pi* by using the global average value, stored in the DDict at key value `"pi"` and computing its own value using the Monte Carlo method. In this way the newly computed value is evenly weighted with the global average. The worker then updates its estimation of *pi* in the DDict.
2. Decide where in the worker code `checkpoint` should be called while computing new values of *pi* in the worker.
3. The global average of *pi* is calculated in the main program using all the local *pi* estimations from each of the workers. Then the global average is stored back in the DDict. 
4. Decide where in the main program code `checkpoint` should be called while updating the global average.
5. Write the code and try it out. Due to the random nature of the program, no two runs will be exactly identical.

In [ ]:
import os
from pathlib import Path
import random

from dragon.native.process_group import ProcessGroup
from dragon.native.process import ProcessTemplate
from dragon.data import DDict
from dragon.utils import XPickler

def worker(ddict, iterations, samples_per_iteration):
    proc_id = ddict.fetch_add("worker_id")

    print(f"Worker {proc_id} says: Hello!", flush=True)
    for i in range(iterations):
        # The following waits until this value is available for this checkpoint
        # because of wait_for_keys on the ddict.
        print(f"Worker {proc_id} waiting for pi from orchestrator.", flush=True)
        pi = ddict["pi"]

        inside_circle = 0
        for _ in range(samples_per_iteration):
            x_coord = random.random()
            y_coord = random.random()
            if x_coord * x_coord + y_coord * y_coord <= 1.0:
                inside_circle += 1

        pi = (pi + (4.0 * inside_circle / samples_per_iteration))/2

        print(f"Proc {proc_id} providing its update for pi={pi}", flush=True)
        ddict[f"worker{proc_id}"] = pi

## Main Program

ddict = DDict(
    1,
    1,
    128 * 1024 * 1024,
    wait_for_keys=True,
    working_set_size=4,
)

samples_per_iteration = 1000
iterations = 7

pg = ProcessGroup()  # or pmi=PMIBackend.PMIX

num_procs = 10
proc_template = ProcessTemplate(target=worker,
            args=(ddict, iterations, samples_per_iteration),
            cwd=os.getcwd())

pg.add_process(nproc=num_procs, template=proc_template)

print("Writing initial pi value before iteration 0.", flush=True)
ddict.bput("pi", 0.0)

pg.init()
pg.start()

for i in range(iterations):
    pi = 0
    for proc_id in range(num_procs):
        pi += ddict[f"worker{proc_id}"]
        
    pi = pi / num_procs

    print(f"Writing updated pi={pi} value after iteration {i}.", flush=True)
    ddict.bput("pi", pi)

pg.join()
pg.close()
print(f"Writing final version of pi={pi}", flush=True)
ddict.destroy()

<details>
<summary><b>▶ Show Answer</b></summary>

```python
import os
from pathlib import Path
import random

from dragon.native.process_group import ProcessGroup
from dragon.native.process import ProcessTemplate
from dragon.data import DDict
from dragon.utils import XPickler

def worker(ddict, iterations, samples_per_iteration):
    proc_id = ddict.fetch_add("worker_id")

    print(f"Worker {proc_id} says: Hello!", flush=True)
    for i in range(iterations):
        # The following waits until this value is available for this checkpoint
        # because of wait_for_keys on the ddict.
        print(f"Worker {proc_id} waiting for pi from orchestrator.", flush=True)
        pi = ddict["pi"]

        inside_circle = 0
        for _ in range(samples_per_iteration):
            x_coord = random.random()
            y_coord = random.random()
            if x_coord * x_coord + y_coord * y_coord <= 1.0:
                inside_circle += 1

        pi = (pi + (4.0 * inside_circle / samples_per_iteration))/2

        print(f"Proc {proc_id} providing its update for pi={pi}", flush=True)
        ddict[f"worker{proc_id}"] = pi
        ddict.checkpoint()

## Main Program

ddict = DDict(
    1,
    1,
    128 * 1024 * 1024,
    wait_for_keys=True,
    working_set_size=4,
)

samples_per_iteration = 1000
iterations = 7

pg = ProcessGroup()  # or pmi=PMIBackend.PMIX

num_procs = 10
proc_template = ProcessTemplate(target=worker,
            args=(ddict, iterations, samples_per_iteration),
            cwd=os.getcwd())

pg.add_process(nproc=num_procs, template=proc_template)

print("Writing initial pi value before iteration 0.", flush=True)
ddict.bput("pi", 0.0)

pg.init()
pg.start()

for i in range(iterations):
    pi = 0
    for proc_id in range(num_procs):
        pi += ddict[f"worker{proc_id}"]

    ddict.checkpoint()

    pi = pi / num_procs

    print(f"Writing updated pi={pi} value after iteration {i}.", flush=True)
    ddict.bput("pi", pi)

pg.join()
pg.close()
print(f"Writing final version of pi={pi}", flush=True)
ddict.destroy()
```

</details>

---

## Exercise 6 - Share data between Python and C++ with `XPickler` and `Serializable`

**Background:** the final step matches the `cpp()` branch of `parallel_pi.py`. Python wraps the `DDict` with `XPickler`, then C++ workers attach with `DDict<Serializable, Serializable>`, read `pi`, compute a local update, write `worker{id}`, and checkpoint.

**Your task:**

1. Complete the C++ loop below.
2. Compile the worker.
3. In Python, wrap the dictionary with `XPickler`, write the initial `pi`, launch 4 workers, gather worker contributions, checkpoint, average, and publish the updated `pi`.
4. Print the final estimate and destroy the dictionary.

In [ ]:
%%writefile worker_ex6.cpp
#include <random>
#include <string>
#include <dragon/dictionary.hpp>
#include <dragon/serializable.hpp>

int main(int argc, char* argv[]) {
    try {
        const char* ser_ddict = argv[1];
        int iterations = std::stoi(argv[2]);
        int samples_per_iteration = std::stoi(argv[3]);
        DDict<Serializable, Serializable> ddict(ser_ddict, nullptr);
        int proc_id = ddict.fetch_add("worker_id");
        for (int i = 0; i < iterations; i++) {
            double pi = 0.0;
            std::random_device rd;
            std::mt19937 gen(rd());
            std::uniform_real_distribution<double> distrib(0.0, 1.0);
            int inside_circle = 0;
            for (int j = 0; j < samples_per_iteration; j++) {
                double x_coord = distrib(gen);
                double y_coord = distrib(gen);
                if (x_coord * x_coord + y_coord * y_coord <= 1.0) inside_circle += 1;
            }
            pi = (pi + (4.0 * inside_circle / samples_per_iteration)) / 2;
            std::string key_str = "worker" + std::to_string(proc_id);
            SerializableString key = key_str;
            // TODO: read pi from the DDict, write this worker's update, and checkpoint.
        }
    } catch (DragonError ex) {
        return 1;
    }
    return 0;
}

In [ ]:
samples_per_iteration = 1000
iterations = 5
num_workers = 4
!g++ $(dragon-config -o) -I ${VIRTUAL_ENV:-$CONDA_PREFIX}/include -std=c++14 -o worker_ex6 worker_ex6.cpp $(dragon-config -l) -ldl
binary_path = TUTORIAL_DIR / "worker_ex6"
ddict = DDict(1, 1, 128 * 1024 * 1024, wait_for_keys=True, working_set_size=4)
xd = ddict
pg = ProcessGroup()
template = ProcessTemplate(target=str(binary_path), args=(xd.serialize(), str(iterations), str(samples_per_iteration)), cwd=os.getcwd())
pg.add_process(nproc=num_workers, template=template)
pg.init()
pg.start()
pi = 0.0
for iteration in range(iterations):
    for worker_id in range(num_workers):
        pass
pg.join()
pg.close()
print(f"final pi = {pi}")
xd.destroy()

<details>
<summary><b>▶ Show Answer</b></summary>

```cpp
#include <random>
#include <string>
#include <dragon/dictionary.hpp>
#include <dragon/serializable.hpp>

int main(int argc, char* argv[]) {
    try {
        const char* ser_ddict = argv[1];
        int iterations = std::stoi(argv[2]);
        int samples_per_iteration = std::stoi(argv[3]);
        DDict<Serializable, Serializable> ddict(ser_ddict, nullptr);
        int proc_id = ddict.fetch_add("worker_id");
        for (int i = 0; i < iterations; i++) {
            double pi = ddict["pi"];
            std::random_device rd;
            std::mt19937 gen(rd());
            std::uniform_real_distribution<double> distrib(0.0, 1.0);
            int inside_circle = 0;
            for (int j = 0; j < samples_per_iteration; j++) {
                double x_coord = distrib(gen);
                double y_coord = distrib(gen);
                if (x_coord * x_coord + y_coord * y_coord <= 1.0) inside_circle += 1;
            }
            pi = (pi + (4.0 * inside_circle / samples_per_iteration)) / 2;
            std::string key_str = "worker" + std::to_string(proc_id);
            SerializableString key = key_str;
            ddict[key] = pi;
            ddict.checkpoint();
        }
    } catch (DragonError ex) {
        return 1;
    }
    return 0;
}
```

```python
samples_per_iteration = 1000
iterations = 5
num_workers = 4
!g++ $(dragon-config -o) -I ${VIRTUAL_ENV:-$CONDA_PREFIX}/include -std=c++14 -o worker_ex6 worker_ex6.cpp $(dragon-config -l) -ldl
binary_path = TUTORIAL_DIR / "worker_ex6"
ddict = DDict(1, 1, 128 * 1024 * 1024, wait_for_keys=True, working_set_size=4)
xp = XPickler()
xd = ddict.pickler(xp, xp)
pg = ProcessGroup()
template = ProcessTemplate(target=str(binary_path), args=(xd.serialize(), str(iterations), str(samples_per_iteration)), cwd=os.getcwd())
pg.add_process(nproc=num_workers, template=template)
xd["pi"] = 0.0
pg.init()
pg.start()
for iteration in range(iterations):
    pi = 0.0
    for worker_id in range(num_workers):
        pi += xd[f"worker{worker_id}"]
    xd.checkpoint()
    pi = pi / num_workers
    xd["pi"] = pi
pg.join()
pg.close()
print(f"final pi = {pi}")
xd.destroy()
```

This is the same mixed-language pattern used by the `cpp()` branch in `parallel_pi.py`.

</details>

---

## Summary

These exercises rebuilt the `parallel_pi.py` workflow in six pieces: Python child attachment, parent/worker coordination with `bput` and `bget`, Python script launch, C++ binary launch, C++-only sharing through `fetch_add`, and finally the full Python/C++ `XPickler` plus `Serializable` handoff. Compare your final answer with `parallel_pi.py` and `worker.cpp` to see the complete reference implementation.